In [15]:
# testing whatever unpkl func we've got: 

import pandas as pd 
import os
import numpy as np 
pkl_dir = '/home/vincent/Senior thesis work/blechRNN-master/data/changepoint_pkl/'
pkl_path = '/home/vincent/Senior thesis work/blechRNN-master/data/changepoint_pkl/tau_frame.pkl'

In [16]:
df = pd.read_pickle(pkl_path)
df

,basename,taste_num,pkl_path,tau,present,tau_std
0,AM25_4Tastes_200807_092703,0,/media/bigdata/firing_space_plot/changepoint_m...,"[[2189, 2767, 3395], [2190, 3391, 3670], [2107...",True,"[[75.99179944202871, 99.88307212474763, 224.09..."
1,AM25_4Tastes_200807_092703,1,/media/bigdata/firing_space_plot/changepoint_m...,"[[2609, 3721, 3840], [2458, 3085, 3209], [2476...",True,"[[99.7562519519959, 105.3448255715961, 69.7845..."
2,AM25_4Tastes_200807_092703,2,/media/bigdata/firing_space_plot/changepoint_m...,"[[2399, 2963, 3442], [2376, 2800, 3405], [2478...",True,"[[66.5855539813255, 138.1941722713012, 206.936..."
3,AM25_4Tastes_200807_092703,3,/media/bigdata/firing_space_plot/changepoint_m...,"[[2572, 3533, 3635], [2510, 3018, 3781], [2501...",True,"[[65.13301711044849, 103.33819824700574, 80.17..."
4,AM25_4Tastes_200806_094914,0,/media/bigdata/firing_space_plot/changepoint_m...,"[[2847, 3769, 3875], [2366, 3663, 3709], [2332...",True,"[[135.27145735496973, 144.12895129012088, 78.6..."
...,...,...,...,...,...,...
67,AM17_4Tastes_191125_084206,3,/media/bigdata/firing_space_plot/changepoint_m...,"[[2072, 2178, 3922], [3549, 3656, 3777], [2912...",True,"[[76.83651852080162, 161.4674047236698, 108.90..."
68,AM17_4Tastes_191126_084934,0,/media/bigdata/firing_space_plot/changepoint_m...,"[[2306, 3100, 3558], [2391, 3589, 3859], [2504...",True,"[[98.77164609016121, 285.0875652303338, 216.88..."
69,AM17_4Tastes_191126_084934,1,/media/bigdata/firing_space_plot/changepoint_m...,"[[2422, 2632, 3835], [2441, 2577, 3618], [2272...",True,"[[124.60341221610877, 175.17639509294656, 198...."
70,AM17_4Tastes_191126_084934,2,/media/bigdata/firing_space_plot/changepoint_m...,"[[2072, 2898, 3748], [2242, 2595, 3532], [2132...",True,"[[61.93171593842289, 145.21460127610297, 188.6..."


In [10]:
df['tau'][0]

array([[2189, 2767, 3395],
       [2190, 3391, 3670],
       [2107, 2830, 3192],
       [2197, 2834, 2982],
       [2073, 3137, 3517],
       [2175, 2499, 2651],
       [2175, 2548, 2708],
       [2348, 3068, 3687],
       [2278, 3043, 3323],
       [2755, 2939, 3842],
       [2452, 2949, 3400],
       [2312, 2754, 2972],
       [2045, 2468, 2576],
       [2828, 3088, 3305],
       [2357, 2798, 3080],
       [2556, 2989, 3812],
       [2563, 2979, 3685],
       [2441, 2829, 3846],
       [2151, 2965, 3438],
       [2553, 2678, 3735],
       [2465, 2654, 3592],
       [2677, 3000, 3577],
       [2529, 3047, 3260],
       [2497, 3296, 3514],
       [2332, 2734, 3031],
       [2261, 2732, 2925],
       [2372, 2515, 2688],
       [2510, 2664, 2766],
       [2488, 2645, 3237],
       [2221, 2889, 3069]])

In [17]:
def extract_valid_changepoints(pkl_path, dataset_num):
    def truncate_name(name):
        numbers = []
        parts = name.split("_")
        truncated_parts = []
        for part in parts:
            if any(char.isdigit() for char in part):
                numbers.extend(char for char in part if char.isdigit())
                truncated_parts.append(part)
        return "_".join(truncated_parts)

    try:
        truncated_dataset_num = truncate_name(dataset_num)
        print(f"Processing dataset number: {truncated_dataset_num}")
        raw_pkl = []
        pkl_files = [f for f in os.listdir(pkl_path) if f.endswith(".pkl")]
        for pkl_file in pkl_files:
            pkl_file_path = os.path.join(pkl_path, pkl_file)
            try:
                df = pd.read_pickle(pkl_file_path)
                for idx, row in df.iterrows():
                    row_basename = os.path.basename(row.iloc[0])
                    truncated_basename = truncate_name(row_basename)
                    if truncated_basename == truncated_dataset_num:
                        extracted_row = row.values
                        if isinstance(extracted_row[3], np.ndarray):
                            raw_pkl.append(extracted_row)
                        else:
                            print(
                                f"Skipping dataset {truncated_dataset_num} due to invalid data structure at index {idx}: {extracted_row[3]}"
                            )
                            raw_pkl = []
                            break
            except Exception as e:
                print(f"Error processing file {pkl_file}: {e}")
                raw_pkl = []
                break

        if not raw_pkl:
            print(
                f"Skipping dataset {truncated_dataset_num} due to invalid data structures."
            )
            return None

        print(
            f"Finished processing dataset {truncated_dataset_num}. Starting to process raw_pkl."
        )
        raw_pkl = np.array(raw_pkl, dtype=object)

        # Create extracted_pkl by ensuring correct length and adding the fourth number
        extracted_pkl = []
        for i in range(len(raw_pkl)):
            new_row = list(raw_pkl[i])
            new_sub_array = []
            for j in range(len(new_row[3])):
                row = new_row[3][j]
                if isinstance(row, (list, np.ndarray)) and len(row) == 3:
                    row = np.append(row, row[2] + 2000)
                elif isinstance(row, (list, np.ndarray)) and len(row) < 3:
                    padding = [np.nan] * (3 - len(row))
                    row = np.append(row, padding)
                    row = np.append(row, row[2] + 2000)
                else:
                    print(
                        f"Unexpected data structure for row {j} in new_row[{i}]: {row}"
                    )
                    continue
                new_sub_array.append(row)
            new_row[3] = np.array(new_sub_array)
            extracted_pkl.append(new_row)

        extracted_pkl = np.array(extracted_pkl, dtype=object)
        print(f"Finished processing valid dataset {truncated_dataset_num}.")
        return extracted_pkl if len(extracted_pkl) > 0 else None
    except FileNotFoundError:
        print(f"Error: Directory {pkl_path} not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [21]:
cp = extract_valid_changepoints(pkl_dir, dataset_num='AM25_4Tastes_200807_092703')

Processing dataset number: AM25_4Tastes_200807_092703
Finished processing dataset AM25_4Tastes_200807_092703. Starting to process raw_pkl.
Finished processing valid dataset AM25_4Tastes_200807_092703.


In [27]:
cp

array([['AM25_4Tastes_200807_092703', '0',
        '/media/bigdata/firing_space_plot/changepoint_mcmc/saved_models/pretty_gc_trans/pretty_gc_trans_727a7a3a',
        array([[2189, 2767, 3395, 5395],
               [2190, 3391, 3670, 5670],
               [2107, 2830, 3192, 5192],
               [2197, 2834, 2982, 4982],
               [2073, 3137, 3517, 5517],
               [2175, 2499, 2651, 4651],
               [2175, 2548, 2708, 4708],
               [2348, 3068, 3687, 5687],
               [2278, 3043, 3323, 5323],
               [2755, 2939, 3842, 5842],
               [2452, 2949, 3400, 5400],
               [2312, 2754, 2972, 4972],
               [2045, 2468, 2576, 4576],
               [2828, 3088, 3305, 5305],
               [2357, 2798, 3080, 5080],
               [2556, 2989, 3812, 5812],
               [2563, 2979, 3685, 5685],
               [2441, 2829, 3846, 5846],
               [2151, 2965, 3438, 5438],
               [2553, 2678, 3735, 5735],
               [2465, 